# AChE Drug Discovery Practical: Structure, Drug-Likeness & Docking (Top 4)
### Target: Acetylcholinesterase (AChE), human, PDB 9OMT - Alzheimer's disease

1. Structure, similarity and drug-likeness of 10 known AChE inhibitors
2. Select the best 4 candidates (not all 10) for docking
3. Identify the active site (P2Rank + known catalytic residues), define the grid box
4. Dock the 4 ligands with AutoDock Vina, validate, and view everything in 3D

Run cells top to bottom. Install cell takes 2-3 minutes.

## Part A - Structure and similarity

### Step 0: Install and import

In [ ]:
!pip install rdkit biopython py3Dmol -q
!apt-get update -qq
!apt-get install -y openbabel default-jre -qq > /dev/null
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64 -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina

In [ ]:
import os, re, glob, subprocess, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import py3Dmol

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from sklearn.decomposition import PCA
from Bio.PDB import MMCIFParser, PDBIO, Select

RDLogger.DisableLog("rdApp.*")
print("Ready.")

### Step 1: Load the 10 known AChE inhibitors (ChEMBL)

In [ ]:
data = {
    "ChEMBL_ID": ["CHEMBL278819", "CHEMBL1201341", "CHEMBL1201349", "CHEMBL1201229",
                  "CHEMBL1652", "CHEMBL2105345", "CHEMBL659", "CHEMBL1104",
                  "CHEMBL502", "CHEMBL1025"],
    "Name": ["Minaprine", "Echothiophate", "Hexafluorenium", "Demecarium",
             "Ambenonium", "Propanidid", "Galantamine", "Edrophonium",
             "Donepezil", "Isoflurophate"],
    "SMILES": [
        "Cc1cc(-c2ccccc2)nnc1NCCN1CCOCC1",
        "CCOP(=O)(OCC)SCC[N+](C)(C)C",
        "C[N+](C)(CCCCCC[N+](C)(C)C1c2ccccc2-c2ccccc21)C1c2ccccc2-c2ccccc21",
        "CN(CCCCCCCCCCN(C)C(=O)Oc1cccc([N+](C)(C)C)c1)C(=O)Oc1cccc([N+](C)(C)C)c1",
        "CC[N+](CC)(CCNC(=O)C(=O)NCC[N+](CC)(CC)Cc1ccccc1Cl)Cc1ccccc1Cl",
        "CCCOC(=O)Cc1ccc(OCC(=O)N(CC)CC)c(OC)c1",
        "COc1ccc2c3c1O[C@H]1C[C@@H](O)C=C[C@@]31CCN(C)C2",
        "CC[N+](C)(C)c1cccc(O)c1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2",
        "CC(C)OP(=O)(F)OC(C)C",
    ],
}
df = pd.DataFrame(data)
df["Mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df.index = range(1, len(df) + 1)
df.index.name = "S.No."

print(f"Loaded {len(df)} molecules (source: ChEMBL database, https://www.ebi.ac.uk/chembl/)")
df[["ChEMBL_ID", "Name", "SMILES"]]

### Step 2: 2D structures

In [ ]:
Draw.MolsToGridImage(df["Mol"].tolist(), molsPerRow=5, subImgSize=(250, 250),
                     legends=df["Name"].tolist())

### Step 3: Fingerprints and similarity
Each molecule becomes a 2048-bit Morgan fingerprint. Tanimoto similarity compares two
fingerprints (1.0 = identical, 0 = nothing in common).

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in df["Mol"]]

n = len(fps)
sim_matrix = np.array([DataStructs.BulkTanimotoSimilarity(fps[i], fps) for i in range(n)])
sim_df = pd.DataFrame(sim_matrix, index=df["Name"], columns=df["Name"])

plt.figure(figsize=(9, 7))
sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1)
plt.title("Tanimoto similarity between the 10 AChE inhibitors")
plt.tight_layout(); plt.show()

pairs = [(sim_df.index[i], sim_df.columns[j], sim_matrix[i, j])
         for i in range(n) for j in range(i + 1, n)]
pairs.sort(key=lambda x: -x[2])
print("Top 5 most similar pairs:")
for a, b, s in pairs[:5]:
    print(f"  {a:15s} - {b:15s}  {s:.2f}")

### Step 4: Chemical space (PCA)

In [ ]:
fp_array = np.array([list(fp) for fp in fps])
pca = PCA(n_components=2)
coords = pca.fit_transform(fp_array)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=100, c="steelblue", edgecolors="black")
for i, name in enumerate(df["Name"]):
    plt.annotate(name, (coords[i, 0], coords[i, 1]), fontsize=9,
                 xytext=(5, 5), textcoords="offset points")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("Chemical space of the 10 AChE inhibitors")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Part B - Drug-likeness (Lipinski + Veber)
**Lipinski:** likely orally absorbed if at most one of MW > 500, LogP > 5, HBD > 5, HBA > 10
is violated. **Veber:** also needs TPSA <= 140 and rotatable bonds <= 10. We also flag
**quaternary ammonium** groups (permanent positive charge) - these cross the blood-brain
barrier poorly, which matters for a brain target like AChE.

In [ ]:
quat = Chem.MolFromSmarts("[NX4+]")

def calc_descriptors(mol):
    return pd.Series({
        "MolWt": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "TPSA": round(Descriptors.TPSA(mol), 2),
        "RotatableBonds": Descriptors.NumRotatableBonds(mol),
        "Quaternary_N": mol.HasSubstructMatch(quat),
    })

df = pd.concat([df, df["Mol"].apply(calc_descriptors)], axis=1)
df["Lipinski_Violations"] = (
    (df["MolWt"] > 500).astype(int) + (df["LogP"] > 5).astype(int) +
    (df["HBD"] > 5).astype(int) + (df["HBA"] > 10).astype(int)
)
df["Drug_Like"] = df["Lipinski_Violations"] <= 1
df["Veber_Pass"] = (df["TPSA"] <= 140) & (df["RotatableBonds"] <= 10)

descriptor_table = df[["Name", "MolWt", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
                       "Lipinski_Violations", "Drug_Like", "Veber_Pass", "Quaternary_N"]]
descriptor_table

## Part C - Selecting the top 4 candidates
Docking all 10 properly takes time, so (as in real pipelines) we shortlist with cheap
filters first and dock only the shortlist. Score (0-4) = Lipinski violations
(0 -> 2 pts, 1 -> 1 pt) + Veber pass (+1) + no quaternary charge (+1). Top 4 go forward.

In [ ]:
def druglikeness_score(row):
    violation_points = {0: 2, 1: 1}.get(row["Lipinski_Violations"], 0)
    return violation_points + int(row["Veber_Pass"]) + int(not row["Quaternary_N"])

df["Selection_Score"] = df.apply(druglikeness_score, axis=1)
ranked = df[["Name", "Lipinski_Violations", "Veber_Pass", "Quaternary_N", "Selection_Score"]] \
         .sort_values("Selection_Score", ascending=False).reset_index(drop=True)
ranked.index = range(1, len(ranked) + 1)
ranked.index.name = "Rank"
print(ranked.to_string())

TOP_N = 4
selected_names = ranked["Name"].head(TOP_N).tolist()
selected_smiles = {name: df.loc[df.Name == name, "SMILES"].iloc[0] for name in selected_names}
print(f"\nSelected for docking: {selected_names}")

## Part D - Docking (top 4 ligands only)

| Step | What | Why |
|---|---|---|
| D1 | Download 9OMT | Get real 3D coordinates of AChE |
| D2 | Clean receptor | Remove water/crystal ligands, protein only |
| D3 | Receptor -> PDBQT | Adds hydrogens + charges, format Vina needs |
| D4 | Active site ID + grid box | P2Rank + known catalytic residues |
| D5-D6 | Ligand prep -> PDBQT | 3D build at pH 7.4, same format conversion |
| D7 | Docking | Vina scores each ligand's best pose |
| D8 | Results | Compare affinity and ligand efficiency |
| D9 | Redocking validation | Sanity-check the whole protocol (RMSD) |
| D10 | 3D view | Protein + docked ligand |

### D1: Download the structure

In [ ]:
PDB_ID = "9OMT"
r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.cif")
open(f"{PDB_ID}.cif", "w").write(r.text)
print(f"Downloaded {PDB_ID}.cif, status {r.status_code}")

### D2: Receptor cleaning - keep protein only (both chains)

In [ ]:
parser = MMCIFParser(QUIET=True)
structure = parser.get_structure(PDB_ID, f"{PDB_ID}.cif")
model = structure[0]
print("Chains:", [c.id for c in model])

standard_aa = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE",
               "LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"}

class ProteinOnly(Select):
    def accept_residue(self, res):
        return res.id[0] == " " and res.get_resname().strip() in standard_aa
    def accept_atom(self, atom):
        return atom.get_altloc() in (" ", "A")

io = PDBIO()
io.set_structure(structure)
io.save("receptor_clean.pdb", ProteinOnly())

# crystal ligand in chain A - kept aside for the grid box check and redocking later
hets_A = {}
for res in model["A"]:
    name = res.get_resname().strip()
    if res.id[0].startswith("H_") and name != "HOH":
        hets_A[f"{name}_{res.id[1]}"] = res
print("Crystal ligand(s) in chain A:", list(hets_A.keys()))

**Quick look:** the parsed structure, both chains, as it came from the PDB.

In [ ]:
view = py3Dmol.view(width=850, height=450)
view.addModel(open("receptor_clean.pdb").read(), "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "skyblue"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "salmon"}})
view.zoomTo()
view.show()

### D3: Format conversion (PDB -> PDBQT)
Adds hydrogens + Gasteiger charges; `-xr` marks this as a rigid receptor.

In [ ]:
!obabel receptor_clean.pdb -O receptor.pdbqt -xr 2>/dev/null
print("Receptor PDBQT ready:", os.path.getsize("receptor.pdbqt") > 0)

### D4: Active site identification (P2Rank + known catalytic residues)

P2Rank predicts pockets from surface shape alone. We run it on **chain A only** - AChE's
two chains are identical copies, so analysing both would just double-count the same
pocket. We then cross-check its top pocket against AChE's catalytic triad
(Ser203, His447, Glu334) and Trp86/Trp286.

In [ ]:
# chain-A-only PDB for P2Rank (one active site, not two copies)
class ProteinChainA(Select):
    def accept_chain(self, c):
        return c.id == "A"
    def accept_residue(self, res):
        return res.id[0] == " " and res.get_resname().strip() in standard_aa
    def accept_atom(self, atom):
        return atom.get_altloc() in (" ", "A")

io.save("receptor_chainA.pdb", ProteinChainA())

!wget -q https://github.com/rdk/p2rank/releases/download/2.4.2/p2rank_2.4.2.tar.gz -O p2rank.tar.gz
!tar -xzf p2rank.tar.gz
p2rank_dir = glob.glob("p2rank_2.4.2*")[0]
!chmod +x {p2rank_dir}/prank
!{p2rank_dir}/prank predict -f receptor_chainA.pdb -o p2rank_output -threads 2

In [ ]:
pred_file = "p2rank_output/receptor_chainA.pdb_predictions.csv"
pockets = pd.read_csv(pred_file, skipinitialspace=True)
pockets.columns = pockets.columns.str.strip()
pockets.index = range(1, len(pockets) + 1)
pockets.index.name = "P2Rank_rank"
print(f"P2Rank predicted {len(pockets)} pocket(s) on chain A.")
pockets[["rank", "score", "probability", "center_x", "center_y", "center_z"]].head(5)

**Pocket 1 and pocket 2**, shown on the protein (gold = pocket 1, teal = pocket 2).

In [ ]:
view = py3Dmol.view(width=850, height=450)
view.addModel(open("receptor_chainA.pdb").read(), "pdb")
view.setStyle({}, {"cartoon": {"color": "lightgrey"}})
for rank, color in [(1, "orange"), (2, "teal")]:
    p = pockets.loc[rank]
    view.addSphere({"center": {"x": p.center_x, "y": p.center_y, "z": p.center_z},
                    "radius": 3, "color": color, "opacity": 0.7})
    view.addLabel(f"Pocket {rank}", {"position": {"x": p.center_x, "y": p.center_y, "z": p.center_z},
                                     "backgroundColor": color, "fontColor": "white"})
view.zoomTo()
view.show()

In [ ]:
def catalytic_matches(residue_ids, resnums=(203, 86, 447, 334)):
    tokens = str(residue_ids).split()
    return sum(1 for rn in resnums if f"A_{rn}" in tokens)

pockets["catalytic_matches"] = pockets["residue_ids"].apply(catalytic_matches)
best_pocket = pockets.sort_values("catalytic_matches", ascending=False).iloc[0]
p2rank_center = np.array([best_pocket.center_x, best_pocket.center_y, best_pocket.center_z])
print(f"Best-matching pocket: P2Rank rank {int(best_pocket['rank'])}, "
     f"matches {int(best_pocket.catalytic_matches)}/4 known catalytic residues")

Combine with the residue-based centre (catalytic triad average) and set the final grid box.

In [ ]:
BOX = np.array([24.0, 24.0, 24.0])

triad_atoms = []
for rn, rid in [("SER", 203), ("HIS", 447), ("GLU", 334)]:
    res = model["A"][rid]
    triad_atoms.extend(a.get_coord() for a in res)
residue_based_center = np.array(triad_atoms).mean(axis=0)

if best_pocket.catalytic_matches >= 2:
    pocket_center = (p2rank_center + residue_based_center) / 2
    print("P2Rank and the known catalytic residues agree - using the average as the final centre.")
else:
    pocket_center = residue_based_center
    print("WARNING: P2Rank's top pocket did not match well - using the residue-based centre only.")

half = BOX / 2
check_residues = {"TRP86": ("TRP", 86), "SER203": ("SER", 203), "HIS447": ("HIS", 447),
                  "GLU334": ("GLU", 334), "TRP286": ("TRP", 286)}
all_inside = True
for label, (rn, rid) in check_residues.items():
    xyz = np.array([a.get_coord() for a in model["A"][rid]])
    inside = bool(np.any(np.all(np.abs(xyz - pocket_center) <= half, axis=1)))
    all_inside = all_inside and inside
    print(f"  {label:7s} inside box: {'YES' if inside else 'NO'}")
print("\n->", "GOOD: box covers the catalytic site." if all_inside else "WARNING: increase BOX.")

**Grid box on the active site**, with the catalytic residues as sticks.

In [ ]:
view = py3Dmol.view(width=850, height=450)
view.addModel(open("receptor_clean.pdb").read(), "pdb")
view.setStyle({}, {"cartoon": {"color": "lightgrey"}})
view.addStyle({"chain": "A", "resi": [86, 203, 334, 447]}, {"stick": {"colorscheme": "orangeCarbon"}})

# label each residue so it's clear these are protein side chains, not a ligand
residue_labels = {86: "Trp86", 203: "Ser203", 334: "Glu334", 447: "His447"}
for resi, label in residue_labels.items():
    for atom in model["A"][resi]:
        if atom.get_name() == "CA":
            c = atom.get_coord()
            view.addLabel(label, {"position": {"x": float(c[0]), "y": float(c[1]), "z": float(c[2])},
                                  "backgroundColor": "0x5C3317", "fontColor": "white", "fontSize": 11})

# draw the box as 12 thick cylinder edges (a wireframe box's lines stay hairline-thin
# in most browsers, no matter what linewidth is set)
cx, cy, cz = pocket_center
hw, hh, hd = BOX[0] / 2, BOX[1] / 2, BOX[2] / 2
corners = [{"x": cx + sx * hw, "y": cy + sy * hh, "z": cz + sz * hd}
          for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)]
edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]
for a, b in edges:
    view.addCylinder({"start": corners[a], "end": corners[b], "radius": 0.15,
                      "color": "0x5C3317", "fromCap": 1, "toCap": 1})

view.zoomTo({"chain": "A", "resi": [86, 203, 334, 447]})
view.zoom(0.4)
view.show()

### D5-D6: Ligand prep + format conversion
Build 3D at pH 7.4, convert to PDBQT (adds hydrogens + charges).

In [ ]:
os.makedirs("ligands", exist_ok=True)

def prepare_ligand(smiles, path):
    subprocess.run(["obabel", f"-:{smiles}", "-O", path, "--gen3d", "-p", "7.4"],
                   capture_output=True, text=True, timeout=120)
    return os.path.exists(path) and os.path.getsize(path) > 0

for name, smi in selected_smiles.items():
    ok = prepare_ligand(smi, f"ligands/{name}.pdbqt")
    print(f"{name:12s} PDBQT ready: {ok}")

### D7: Dock with AutoDock Vina

In [ ]:
EXHAUSTIVENESS = 16
NUM_MODES = 5

def run_vina(ligand, out, center, box=BOX, exh=EXHAUSTIVENESS, seed=42):
    cmd = ["vina", "--receptor", "receptor.pdbqt", "--ligand", ligand,
           "--center_x", f"{center[0]:.3f}", "--center_y", f"{center[1]:.3f}", "--center_z", f"{center[2]:.3f}",
           "--size_x", f"{box[0]}", "--size_y", f"{box[1]}", "--size_z", f"{box[2]}",
           "--exhaustiveness", str(exh), "--num_modes", str(NUM_MODES), "--seed", str(seed), "--out", out]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None, (result.stderr or result.stdout)[-300:]
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", result.stdout, re.M)
    return (float(match.group(1)) if match else None), ""

def heavy_atom_count(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumHeavyAtoms() if mol else None

results = []
for name, smi in selected_smiles.items():
    affinity, err = run_vina(f"ligands/{name}.pdbqt", f"ligands/{name}_out.pdbqt", pocket_center)
    if affinity is None:
        print(f"{name:12s} FAILED ({err[:100]})")
        continue
    heavy = heavy_atom_count(smi)
    results.append({"Name": name, "Affinity_kcal_mol": affinity, "Heavy_atoms": heavy,
                    "Ligand_efficiency": round(-affinity / heavy, 3)})
    print(f"{name:12s} {affinity:.2f} kcal/mol")

results_df = pd.DataFrame(results).sort_values("Affinity_kcal_mol").reset_index(drop=True)
results_df.index = range(1, len(results_df) + 1)
results_df.index.name = "Rank"
results_df

### D8: Results
More negative affinity = stronger predicted binding. Ligand efficiency (affinity / heavy atoms) corrects for molecule size.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].barh(results_df["Name"], results_df["Affinity_kcal_mol"], color="crimson")
ax[0].invert_yaxis(); ax[0].set_xlabel("kcal/mol"); ax[0].set_title("Docking score")

le = results_df.sort_values("Ligand_efficiency", ascending=False)
ax[1].barh(le["Name"], le["Ligand_efficiency"], color="seagreen")
ax[1].invert_yaxis(); ax[1].set_xlabel("kcal/mol per heavy atom"); ax[1].set_title("Ligand efficiency")
plt.tight_layout(); plt.show()

### D9: Redocking validation
Dock the real crystal ligand back into its own pocket and measure RMSD to its crystal
position. RMSD < 2 A means the protocol can reproduce a known binding mode. We keep the
ligand's **original crystal coordinates** here (no `--gen3d`) - regenerating 3D from
scratch was the cause of the previous PDBQT error.

In [ ]:
ref_key = max(hets_A, key=lambda k: len(hets_A[k]))
ref_resnum = int(ref_key.split("_")[-1])
print(f"Reference ligand: {ref_key}")

# Plain PDB has no CONECT records for ligands, so OpenBabel must guess bonds from
# distances alone - that guess was wrong for this molecule (a long bis-oxime
# reactivator) and produced a broken "multiple ROOT" PDBQT. RCSB's ModelServer
# instead returns this exact ligand copy, correct bonds included, as SDF.
sdf_url = f"https://models.rcsb.org/v1/{PDB_ID.lower()}/ligand?auth_asym_id=A&auth_seq_id={ref_resnum}&encoding=sdf"
open("ref_ligand.sdf", "w").write(requests.get(sdf_url).text)
subprocess.run(["obabel", "ref_ligand.sdf", "-O", "ref_ligand.pdbqt", "-h"],
               capture_output=True, text=True, timeout=60)

def heavy_coords(path, first_model_only=True):
    coords = []
    for line in open(path):
        if line.startswith("ENDMDL") and first_model_only and coords:
            break
        if line.startswith(("ATOM", "HETATM")) and line[77:79].strip() not in ("H", "HD", "HS"):
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(coords)

try:
    if not (os.path.exists("ref_ligand.pdbqt") and os.path.getsize("ref_ligand.pdbqt") > 0):
        raise RuntimeError("OpenBabel could not build a PDBQT file for the crystal ligand.")
    aff, err = run_vina("ref_ligand.pdbqt", "ref_redock_out.pdbqt", pocket_center)
    if aff is None:
        raise RuntimeError(err)
    a, b = heavy_coords("ref_ligand.pdbqt"), heavy_coords("ref_redock_out.pdbqt")
    if len(a) != len(b):
        raise RuntimeError(f"Atom count mismatch ({len(a)} vs {len(b)}).")
    rmsd = float(np.sqrt(((a - b) ** 2).sum(axis=1).mean()))
    print(f"Redocking score: {aff:.2f} kcal/mol")
    print(f"RMSD to crystal pose: {rmsd:.2f} A ->",
         "GOOD (< 2 A)." if rmsd < 2 else "POOR (> 2 A): treat scores with extra care.")
except Exception as e:
    print(f"Redocking validation failed: {e}")
    print("Docking runs above are still valid - this is just one extra sanity check.")

### D10: Protein + docked ligand in 3D

In [ ]:
def first_model_pdb(path_pdbqt):
    lines = []
    for l in open(path_pdbqt):
        if l.startswith("ENDMDL"):
            break
        lines.append(l)
    open("tmp_pose.pdbqt", "w").write("".join(lines))
    subprocess.run(["obabel", "tmp_pose.pdbqt", "-O", "tmp_pose.pdb"], capture_output=True)
    return open("tmp_pose.pdb").read()

def show_pose(name):
    score = results_df.loc[results_df.Name == name, "Affinity_kcal_mol"].iloc[0]
    view = py3Dmol.view(width=850, height=500)
    view.addModel(open("receptor_clean.pdb").read(), "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
    view.addStyle({"model": 0, "chain": "A", "resi": [86, 203, 334, 447]},
                  {"stick": {"colorscheme": "orangeCarbon"}})
    view.addModel(first_model_pdb(f"ligands/{name}_out.pdbqt"), "pdb")
    view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
    view.zoomTo({"model": 1})
    print(f"{name}: {score:.2f} kcal/mol")
    view.show()

for name in results_df["Name"]:
    show_pose(name)

In [ ]:
results_df.to_csv("docking_results_top4.csv", index=False)
try:
    from google.colab import files
    files.download("docking_results_top4.csv")
except Exception:
    print("Saved docking_results_top4.csv")